In [52]:
import pandas as pd
import numpy as np

# train_test_split: splits data into training and testing parts
from sklearn.model_selection import train_test_split

# RobustScaler: a scaler that is not thrown off by extreame values
from sklearn.preprocessing import RobustScaler

# joblib: lets us save Pytho objets ( like the scaler ) to a file
import joblib

pd.set_option("display.max_columns",None)

In [53]:
# Load the raw dataset
df = pd.read_csv("../data/creditcard.csv")
print("Original shape:", df.shape)

Original shape: (284807, 31)


Remove duplicates

In [54]:
# Count duplicates rows before removing them
print("Duplicates before:", df.duplicated().sum())

# drop_duplicates() keeps the first copy of each row and removes the rest
df = df.drop_duplicates()

# reset_index() renumbers row 0,1,2,3,... after rows were removed
df = df.reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

# Check the class counts again; a few fraud rows were duplicates too
print(df["Class"].value_counts())

Duplicates before: 1081
Shape after removing duplicates: (283726, 31)
Class
0    283253
1       473
Name: count, dtype: int64


Separate features and target

In [55]:
# X = the input features (everything except Class )
X = df.drop(columns=["Class"])

# y = the target we want to predict ( 0 = normal, 1 = fraud)
y = df["Class"]

print("X shape:", X.shape)  # (rows, 30)
print("y shape:", y.shape)  # (rows,)

X shape: (283726, 30)
y shape: (283726,)


Train/test split

In [56]:
# Split into 80% training and 20% testing
# test_size=0.2     -> 20% of rows go to the test set
# random_state=42   -> makes the split the same every time you run it (reproducible)
# stratify=y        -> keeps the same fraud percentage in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (226980, 30)
X_test:  (56746, 30)


Why stratify=y matters: fraud is only about 0.17% of the data. Without stratification, a random split could put very few fraud cases in the test set, or even none.

verify the split

In [57]:
# Check the fraud percentage in each part; both should be about 0.17%
print("Fraud % in train:",round(y_train.mean() * 100,3))
print("Fraud % in test:",round(y_test.mean() * 100,3))

# Count of fraud cases in each part
print("Fraud cases in train:",y_train.sum())
print("Fraud cases in test:",y_test.sum())
#Expected: about 378 fraud cases in train and about 95 in test.

Fraud % in train: 0.167
Fraud % in test: 0.167
Fraud cases in train: 378
Fraud cases in test: 95


In [58]:
# The two columns that need scalling
cols_to_scale = ["Time","Amount"]

# Make copies so we don't modify the original data by accident
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Create the scaler
scaler = RobustScaler()

# fit_transform on TRAIN : the scaler learns the median/IQR from training data only.
# then applies the scaling
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

# transform on TEST: uses the numbers learned from TRAIN ( never fit on test data!)
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# This is the key line to understand:
# fit only on train, transform on both.
# The test set must act like brand-new data the model has never seen.


check the result

In [59]:
# After scaling, Time and Amount should be centered around 0
print(X_train_scaled[cols_to_scale].describe())

                Time         Amount
count  226980.000000  226980.000000
mean        0.118467       0.918890
std         0.557881       3.405863
min        -0.996399      -0.305987
25%        -0.359179      -0.227134
50%         0.000000       0.000000
75%         0.640821       0.772866
max         1.033522     272.096037


Save everything as CSV files

In [60]:
# Save the train/test data as CSV files in data/processed
X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Save the fitted scaler to the models folder.
# We'll need it later to scale brand-new transactions the same way.
joblib.dump(scaler, "../models/scaler.pkl")

print("Saved all files.")

Saved all files.
